In [2]:
import ollama
import pandas as pd 



In [3]:
candidate_prompt = """
You are the Candidate in a job contract negotiation.

Private constraints:
- Target salary: 90,000 USD
- Minimum acceptable salary: 85,000 USD
- Preferred working hours: 8 hours
- Maximum acceptable working hours: 9 hours

Rules:
- Never accept salary below 85,000 USD.
- Never accept working hours above 9.
- If the employer offers salary >= 85,000 and working hours <= 9, accept.
- Otherwise continue negotiating or quit.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [4]:
employer_prompt = """
You are the Employer in a job contract negotiation.

Private constraints:
- Preferred salary offer: 75,000 USD
- Maximum salary offer: 87,000 USD
- Preferred working hours: 10 hours
- Minimum acceptable working hours: 9 hours

Rules:
- Never offer more than 87,000 USD.
- Never accept working hours below 9.
- If candidate asks for salary <= 87,000 and working hours >= 9, accept.
- Otherwise continue negotiating or quit.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [5]:
import re

def parse_structured_response(text):
    result = {
        "message": None,
        "salary_offer": None,
        "hours_offer": None,
        "decision": None
    }

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("MESSAGE:"):
            result["message"] = line.replace("MESSAGE:", "").strip()

        elif line.startswith("SALARY_OFFER:"):
            value = line.replace("SALARY_OFFER:", "").strip()

            if value.upper() == "NONE":
                result["salary_offer"] = None
            else:
                number = re.search(r"\d[\d,]*", value)
                if number:
                    result["salary_offer"] = int(number.group().replace(",", ""))

        elif line.startswith("HOURS_OFFER:"):
            value = line.replace("HOURS_OFFER:", "").strip()

            if value.upper() == "NONE":
                result["hours_offer"] = None
            else:
                number = re.search(r"\d+(\.\d+)?", value)
                if number:
                    result["hours_offer"] = float(number.group())

        elif line.startswith("DECISION:"):
            result["decision"] = line.replace("DECISION:", "").strip().lower()

    return result

In [6]:
def check_agreement_from_offer(speaker, salary, hours):

    if salary is None or hours is None:
        return False

    if speaker == "Employer":
        # L'Employer fa un'offerta.
        # Controlliamo se è accettabile per il Candidate.
        return salary >= 85000 and hours <= 9

    if speaker == "Candidate":
        # Il Candidate fa una proposta.
        # Controlliamo se è accettabile per l'Employer.
        return salary <= 87000 and hours >= 9

    return False

In [7]:
current_message = """
MESSAGE: I would like a salary of 90,000 USD and an 8 hour workday.
SALARY_OFFER: 90000
HOURS_OFFER: 8
DECISION: continue
"""

In [8]:
conversation_log = []

max_turns = 6
outcome = None

for turn in range(max_turns):

    # EMPLOYER TURN
    employer_response = ollama.chat(
        model="llama3",
        messages=[
            {"role": "system", "content": employer_prompt},
            {"role": "user", "content": current_message}
        ]
    )

    employer_text = employer_response["message"]["content"]
    print("\nEMPLOYER:")
    print(employer_text)

    parsed_employer = parse_structured_response(employer_text)

    conversation_log.append({
        "turn": turn,
        "speaker": "Employer",
        "text": employer_text,
        "salary_offer": parsed_employer["salary_offer"],
        "hours_offer": parsed_employer["hours_offer"],
        "decision": parsed_employer["decision"]
    })

    if check_agreement_from_offer(
        "Employer",
        parsed_employer["salary_offer"],
        parsed_employer["hours_offer"]
    ):
        outcome = "Agreement"
        print("\nAGREEMENT REACHED")
        break

    if parsed_employer["decision"] == "quit":
        outcome = "Failure"
        print("\nNEGOTIATION FAILED")
        break

    current_message = employer_text


    # CANDIDATE TURN
    candidate_response = ollama.chat(
        model="llama3",
        messages=[
            {"role": "system", "content": candidate_prompt},
            {"role": "user", "content": current_message}
        ]
    )

    candidate_text = candidate_response["message"]["content"]
    print("\nCANDIDATE:")
    print(candidate_text)

    parsed_candidate = parse_structured_response(candidate_text)

    conversation_log.append({
        "turn": turn,
        "speaker": "Candidate",
        "text": candidate_text,
        "salary_offer": parsed_candidate["salary_offer"],
        "hours_offer": parsed_candidate["hours_offer"],
        "decision": parsed_candidate["decision"]
    })

    if check_agreement_from_offer(
        "Candidate",
        parsed_candidate["salary_offer"],
        parsed_candidate["hours_offer"]
    ):
        outcome = "Agreement"
        print("\nAGREEMENT REACHED")
        break

    if parsed_candidate["decision"] == "quit":
        outcome = "Failure"
        print("\nNEGOTIATION FAILED")
        break

    current_message = candidate_text


if outcome is None:
    outcome = "Timeout"

print("\nFINAL OUTCOME:", outcome)


EMPLOYER:
MESSAGE: That's not within our budget. We're willing to offer up to 87,000 USD and require a minimum of 9 hours worked per day.
SALARY_OFFER: 87000
HOURS_OFFER: 9
DECISION: continue

AGREEMENT REACHED

FINAL OUTCOME: Agreement


In [9]:
simulation_df = pd.DataFrame(conversation_log)
simulation_df

,turn,speaker,text,salary_offer,hours_offer,decision
0,0,Employer,MESSAGE: That's not within our budget. We're w...,87000,9.0,continue


In [10]:
n_runs = 3
max_turns = 6
model_name = "llama3"

In [11]:
import json
import os
import pandas as pd
import ollama

base_path = r"C:\Users\simon\OneDrive\Desktop\negotiation_arena"

scenarios_file = os.path.join(
    base_path,
    "data",
    "scenarios",
    "job_negotiation_scenarios.json"
)

conditions_file = os.path.join(
    base_path,
    "data",
    "scenarios",
    "experimental_conditions.json"
)

with open(scenarios_file, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

with open(conditions_file, "r", encoding="utf-8") as f:
    conditions = json.load(f)

In [12]:
def build_candidate_prompt(style):
    return f"""
You are the Candidate in a job contract negotiation.

Private constraints:
- Target salary: 90,000 USD
- Minimum acceptable salary: 85,000 USD
- Preferred working hours: 8 hours
- Maximum acceptable working hours: 9 hours

Negotiation style: {style}

Rules:
- Never accept salary below 85,000 USD.
- Never accept working hours above 9.
- If the employer offers salary >= 85,000 and working hours <= 9, accept.
- Otherwise continue negotiating or quit.
- Reply concisely.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [13]:
def build_employer_prompt(style):
    return f"""
You are the Employer in a job contract negotiation.

Private constraints:
- Preferred salary offer: 75,000 USD
- Maximum salary offer: 87,000 USD
- Preferred working hours: 10 hours
- Minimum acceptable working hours: 9 hours

Negotiation style: {style}

Rules:
- Never offer more than 87,000 USD.
- Never accept working hours below 9.
- If candidate asks for salary <= 87,000 and working hours >= 9, accept.
- Otherwise continue negotiating or quit.
- Reply concisely.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [14]:
def run_negotiation_simulation(
    scenario,
    condition,
    run_id,
    model_name="llama3",
    max_turns=6
):
    conversation_log = []

    candidate_prompt = build_candidate_prompt(condition["candidate_style"])
    employer_prompt = build_employer_prompt(condition["employer_style"])

    current_message = """
MESSAGE: I would like a salary of 90,000 USD and an 8 hour workday.
SALARY_OFFER: 90000
HOURS_OFFER: 8
DECISION: continue
"""

    outcome = None

    for turn in range(max_turns):

        # Employer turn
        employer_response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": employer_prompt},
                {"role": "user", "content": current_message}
            ]
        )

        employer_text = employer_response["message"]["content"]
        parsed_employer = parse_structured_response(employer_text)

        conversation_log.append({
            "scenario_id": scenario["scenario_id"],
            "condition": condition["condition_name"],
            "run_id": run_id,
            "turn": turn,
            "speaker": "Employer",
            "text": employer_text,
            "salary_offer": parsed_employer["salary_offer"],
            "hours_offer": parsed_employer["hours_offer"],
            "decision": parsed_employer["decision"]
        })

        if check_agreement_from_offer(
            "Employer",
            parsed_employer["salary_offer"],
            parsed_employer["hours_offer"]
        ):
            outcome = "Agreement"
            break

        if parsed_employer["decision"] == "quit":
            outcome = "Failure"
            break

        current_message = employer_text

        # Candidate turn
        candidate_response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": candidate_prompt},
                {"role": "user", "content": current_message}
            ]
        )

        candidate_text = candidate_response["message"]["content"]
        parsed_candidate = parse_structured_response(candidate_text)

        conversation_log.append({
            "scenario_id": scenario["scenario_id"],
            "condition": condition["condition_name"],
            "run_id": run_id,
            "turn": turn,
            "speaker": "Candidate",
            "text": candidate_text,
            "salary_offer": parsed_candidate["salary_offer"],
            "hours_offer": parsed_candidate["hours_offer"],
            "decision": parsed_candidate["decision"]
        })

        if check_agreement_from_offer(
            "Candidate",
            parsed_candidate["salary_offer"],
            parsed_candidate["hours_offer"]
        ):
            outcome = "Agreement"
            break

        if parsed_candidate["decision"] == "quit":
            outcome = "Failure"
            break

        current_message = candidate_text

    if outcome is None:
        outcome = "Timeout"

    return conversation_log, outcome

In [15]:
all_turns = []
all_outcomes = []

for scenario in scenarios:
    for condition in conditions:
        for run_id in range(n_runs):

            print(
                f"Running scenario {scenario['scenario_id']} | "
                f"{condition['condition_name']} | run {run_id}"
            )

            log, outcome = run_negotiation_simulation(
                scenario=scenario,
                condition=condition,
                run_id=run_id,
                model_name=model_name,
                max_turns=max_turns
            )

            all_turns.extend(log)

            all_outcomes.append({
                "scenario_id": scenario["scenario_id"],
                "condition": condition["condition_name"],
                "run_id": run_id,
                "outcome": outcome,
                "n_turns": len(log)
            })

Running scenario 1 | cooperative | run 0
Running scenario 1 | cooperative | run 1
Running scenario 1 | cooperative | run 2
Running scenario 1 | competitive | run 0
Running scenario 1 | competitive | run 1
Running scenario 1 | competitive | run 2
Running scenario 1 | mixed | run 0
Running scenario 1 | mixed | run 1
Running scenario 1 | mixed | run 2
Running scenario 2 | cooperative | run 0
Running scenario 2 | cooperative | run 1
Running scenario 2 | cooperative | run 2
Running scenario 2 | competitive | run 0
Running scenario 2 | competitive | run 1
Running scenario 2 | competitive | run 2
Running scenario 2 | mixed | run 0
Running scenario 2 | mixed | run 1
Running scenario 2 | mixed | run 2
Running scenario 3 | cooperative | run 0
Running scenario 3 | cooperative | run 1
Running scenario 3 | cooperative | run 2
Running scenario 3 | competitive | run 0
Running scenario 3 | competitive | run 1
Running scenario 3 | competitive | run 2
Running scenario 3 | mixed | run 0
Running scenario 

In [16]:
llm_turns_df = pd.DataFrame(all_turns)
llm_outcomes_df = pd.DataFrame(all_outcomes)

In [17]:
llm_outcomes_df

,scenario_id,condition,run_id,outcome,n_turns
0,1,cooperative,0,Agreement,2
1,1,cooperative,1,Agreement,2
2,1,cooperative,2,Agreement,1
3,1,competitive,0,Agreement,2
4,1,competitive,1,Agreement,6
5,1,competitive,2,Agreement,1
6,1,mixed,0,Agreement,1
7,1,mixed,1,Agreement,12
8,1,mixed,2,Agreement,1
9,2,cooperative,0,Agreement,6


In [18]:
llm_outcomes_df["outcome"].value_counts()

outcome
Agreement    27
Name: count, dtype: int64

In [19]:
llm_outcomes_df.groupby("condition")["outcome"].value_counts()

condition    outcome  
competitive  Agreement    9
cooperative  Agreement    9
mixed        Agreement    9
Name: count, dtype: int64

In [20]:
llm_outcomes_df.groupby("condition")["n_turns"].mean()

condition
competitive    2.333333
cooperative    3.333333
mixed          4.444444
Name: n_turns, dtype: float64

Negotiation style significantly affected interaction dynamics. Cooperative agents converged rapidly toward agreements, while competitive and mixed negotiation styles produced longer interactions and occasional negotiation failures.


In [21]:
simulations_path = os.path.join(base_path, "data", "simulations")
os.makedirs(simulations_path, exist_ok=True)

llm_turns_df.to_csv(
    os.path.join(simulations_path, "llm_turns.csv"),
    index=False
)

llm_outcomes_df.to_csv(
    os.path.join(simulations_path, "llm_outcomes.csv"),
    index=False
)

print("Saved simulation results.")

Saved simulation results.
